# LMP inexact low-mode → exact eigenvectors 정제

**설계 원칙**
- D 는 `mul!(y, D, x)` 형태 그대로 — 행렬로 변환하지 않음 (200k × 200k × 16 B ≈ 600 GB)
- 고유벡터도 WF3x1 형태 그대로 — CG 내부에서 `mul!` 만 사용
- 내적/벡터 연산: `inner_product`, `axpby!`

**알고리즘 파이프라인**

| 단계 | 방법 | 역할 |
|------|------|------|
| 1 | LMP Ritz (모델 추론) | Inexact 초기 근사 벡터 |
| 2 | LOPCG | Rayleigh quotient 최소화 → warm-start 생성 |
| 3 | **Block Inverse Iteration** | 내장 `cg(X, DdagD, V; eps=1e-24)` 로 정제 → ‖r‖ < 1e-11 |

**왜 Power iteration / 단순 반복이 안 되는가?**  
D†D 를 반복 적용하면 가장 큰 고유값(λ ≈ 4~5, bulk) 으로 수렴함.  
LOPCG 는 Rayleigh quotient 를 직접 최소화하여 가장 작은 고유값 방향으로 이동.  
Block Inverse Iteration 은 (D†D)⁻¹ 을 CG 로 근사 → 작은 고유값 방향을 지수적으로 증폭.

In [1]:
using LatticeDiracOperators
using Gaugefields
using LMP
using LinearAlgebra
using JLD2
using Printf
using KrylovKit

[ Info: Precompiling LMP [a75e356b-bbea-40ef-9c9d-31e1efba0018] 
┌ Info: Skipping precompilation due to precompilable error. Importing LMP [a75e356b-bbea-40ef-9c9d-31e1efba0018].
└   exception = Error when precompiling module, potentially caused by a __precompile__(false) declaration in the module.
┌ Error: You are using CUDA 13.1.0, but CUDA.jl was precompiled for CUDA 13.0.0.
│ This is unexpected; please file an issue.
└ @ CUDA C:\Users\_s2220186\.julia\packages\CUDA\724Sm\src\initialization.jl:148
[ Info: Threads backend loaded with 1 threads


## 1. 파라미터

In [2]:
NX = 8; NY = 8; NZ = 8; NT = 16
Nwing = 0; NC = 3

params = Dict(
    "Dirac_operator" => "Wilson",
    "κ"              => 0.16,
    "eps_CG"         => 1.0e-8,
    "verbose_level"  => 1,
)

GAUGE_FILE  = "C:/Users/_s2220186/Desktop/Julia/gauge/config_1.jld2"
MODEL_FILE  = "C:/Users/_s2220186/Desktop/multilowmode/MG-testvector/32PTLayerRitz(mass016)(30000epoch)(xavier_init)(4eig)(120config).jld2"
OUTPUT_FILE = "exact_lowmodes_config1.jld2"

# println("Lattice: $(NX)×$(NY)×$(NZ)×$(NT)  κ=$(params[\"κ\"])")

"exact_lowmodes_config1.jld2"

## 2. 기본 연산 정의 (WF3x1 기반)

In [3]:
# D†D x — 연산자는 mul! 형태 그대로
function apply_DdagD!(y, D, x, tmp)
    mul!(tmp, D, x)
    mul!(y, D', tmp)
end

wf_zero!(v)      = fill!(v.f, 0)
wf_copy!(d, s)   = copyto!(d.f, s.f)
wf_norm(v)       = sqrt(real(inner_product(v, v)))

function wf_normalize!(v)
    nrm = wf_norm(v)
    v.f .*= (1 / nrm)
    return nrm
end

# Modified Gram-Schmidt. 수렴한 벡터(norm < thr)는 건너뜀
function mgs!(V; thr=1e-12)
    for i in eachindex(V)
        for j in 1:i-1
            c = inner_product(V[j], V[i])
            axpby!(-c, V[j], 1, V[i])
        end
        n = wf_norm(V[i])
        n > thr ? (V[i].f .*= 1/n) : nothing   # 이미 수렴한 방향은 스킵
    end
    return V
end

println("Primitives defined.")

Primitives defined.


In [4]:
"""
    rayleigh_ritz_k(D, S, AS, k, buf) → (vals[1:k], vecs[1:k], AV[1:k])

직교정규 기저 S 위에서 D†D 를 Rayleigh-Ritz 투영, k 번째로 작은 고유쌍을 반환.
AS[j] = D†D·S[j] 는 호출 전에 미리 계산해 넣는다 (재사용 가능).
D 는 연산자 형태 그대로 — 여기서는 이미 계산된 AS 만 쓰임.
"""
function rayleigh_ritz_k(S, AS, k)
    n = length(S)
    H = Matrix{ComplexF64}(undef, n, n)
    for i in 1:n, j in 1:n
        H[i,j] = inner_product(S[i], AS[j])
    end
    H = Hermitian((H + H') / 2)

    F = eigen(H)                      # n×n — D†D 행렬 아님, 투영 계수 행렬
    C = F.vectors[:, 1:k]             # 가장 작은 k 개 고유벡터 계수
    vals = real.(F.values[1:k])

    psi  = [deepcopy(S[1])  for _ in 1:k]
    Apsi = [deepcopy(AS[1]) for _ in 1:k]
    for m in 1:k
        wf_zero!(psi[m]);  wf_zero!(Apsi[m])
        for j in 1:n
            axpby!(C[j,m], S[j],  1, psi[m])
            axpby!(C[j,m], AS[j], 1, Apsi[m])
        end
        nrm = wf_normalize!(psi[m])
        Apsi[m].f .*= 1/nrm           # AV 도 같은 스케일 유지
    end
    return vals, psi, Apsi
end

println("rayleigh_ritz_k defined.")

rayleigh_ritz_k defined.


In [5]:
const _cg = LatticeDiracOperators.Dirac_operators.cg   # 서브모듈 직접 참조

"""
    precond_lopcg_wf!(DdagD, V; max_iter, prec_eps, prec_steps, tol, verbose)

Preconditioned LOPCG — 내장 cg 로 (D†D)⁻¹ 를 전처리기로 사용.

탐색 방향:
  W_j = D†D·V_j − λ_j V_j          (잔차)
  Z_j ← _cg(DdagD, W_j; eps=prec_eps)  (전처리: (D†D)⁻¹W_j)

비전처리 LOPCG 대비:
  비전처리: 수렴률 ≈ 0.994/iter → 수천 회 필요
  전처리:   (D†D)⁻¹ 가 bulk 모드 억제 → 수십∼백 회면 수렴
"""
function precond_lopcg_wf!(DdagD, V;
        max_iter   = 300,
        prec_eps   = 1e-6,    # ‖r‖² < 1e-6 → ‖r‖ < 1e-3 (느슨해도 됨)
        prec_steps = 200,
        tol        = 1e-11,
        verbose    = true)

    k   = length(V)
    mgs!(V)

    AV = [deepcopy(v) for v in V]
    for j in 1:k
        mul!(AV[j], DdagD, V[j])
    end
    _, V[:], AV[:] = rayleigh_ritz_k(V, AV, k)
    vals = [real(inner_product(V[j], AV[j])) for j in 1:k]

    W     = [deepcopy(v) for v in V]
    Z     = [deepcopy(v) for v in V]
    AZ    = [deepcopy(v) for v in V]
    r_tmp = deepcopy(V[1])
    vb    = Verbose_print(0)

    for iter in 1:max_iter
        # 1. 잔차 W_j = D†D V_j − λ_j V_j
        for j in 1:k
            wf_copy!(W[j], AV[j])
            axpby!(-vals[j], V[j], 1, W[j])
        end

        # 2. 전처리 CG: D†D Z_j = W_j
        for j in 1:k
            wf_copy!(Z[j], W[j])
            Z[j].f .*= 1 / max(vals[j], 1e-14)   # warm-start: Z₀ = W/λ
            _cg(Z[j], DdagD, W[j]; eps=prec_eps, maxsteps=prec_steps, verbose=vb)
        end

        # 3. Z ⊥ V
        for j in 1:k, i in 1:k
            c = inner_product(V[i], Z[j])
            axpby!(-c, V[i], 1, Z[j])
        end
        mgs!(Z; thr=1e-14)

        # 4. D†D·Z
        for j in 1:k
            mul!(AZ[j], DdagD, Z[j])
        end

        # 5. Rayleigh-Ritz on span(V, Z)
        vals_new, psi, Apsi = rayleigh_ritz_k([V..., Z...], [AV..., AZ...], k)
        for j in 1:k
            wf_copy!(V[j],  psi[j])
            wf_copy!(AV[j], Apsi[j])
        end
        vals = vals_new

        # 6. 수렴 판정
        max_res = 0.0
        for j in 1:k
            wf_copy!(r_tmp, AV[j])
            axpby!(-vals[j], V[j], 1, r_tmp)
            max_res = max(max_res, wf_norm(r_tmp))
        end
        if verbose && (iter == 1 || iter % 10 == 0)
            @printf("[P-LOPCG %4d]  λ₁ = %.10e   ‖r‖_max = %.3e\n",
                iter, vals[1], max_res)
        end
        if max_res < tol
            verbose && @printf("Converged at iter %d  ‖r‖_max = %.3e\n",
                iter, max_res)
            return vals, V
        end
    end
    @warn "max_iter=$max_iter 도달  (‖r‖ 확인 필요)"
    return vals, V
end

println("precond_lopcg_wf! defined.")

precond_lopcg_wf! defined.


In [6]:
# WilsonFermion ↔ flat Vector{ComplexF64} — KrylovKit 인터페이스용
# D†D 행렬은 저장하지 않음; eigenvector만 flat 형태로 변환 (각 ~3 MB)
function wf_to_vec(x)
    NX=x.NX; NY=x.NY; NZ=x.NZ; NT=x.NT; NC=x.NC; NG=4
    v = Vector{ComplexF64}(undef, NC*NX*NY*NZ*NT*NG)
    idx = 0
    for ig=1:NG, it=1:NT, iz=1:NZ, iy=1:NY, ix=1:NX, ic=1:NC
        idx += 1
        @inbounds v[idx] = x[ic,ix,iy,iz,it,ig]
    end
    return v
end

function vec_to_wf!(x, v)
    NX=x.NX; NY=x.NY; NZ=x.NZ; NT=x.NT; NC=x.NC; NG=4
    idx = 0
    for ig=1:NG, it=1:NT, iz=1:NZ, iy=1:NY, ix=1:NX, ic=1:NC
        idx += 1
        @inbounds x[ic,ix,iy,iz,it,ig] = v[idx]
    end
    return x
end

println("wf_to_vec / vec_to_wf! defined.")

wf_to_vec / vec_to_wf! defined.


In [7]:
"""
    block_inverse_iter!(DdagD, V; max_iter, cg_eps, tol, verbose)

블록 역반복법 — LatticeDiracOperators 내장 `_cg` 사용.
"""
function block_inverse_iter!(DdagD, V;
        max_iter = 50,
        cg_eps   = 1e-22,
        cg_steps = 5000,
        tol      = 1e-11,
        verbose  = true)

    k   = length(V)
    mgs!(V)

    AV = [deepcopy(v) for v in V]
    for j in 1:k
        mul!(AV[j], DdagD, V[j])
    end
    vals  = [real(inner_product(V[j], AV[j])) for j in 1:k]
    X     = [deepcopy(v) for v in V]
    r_tmp = deepcopy(V[1])
    vb    = Verbose_print(0)

    for iter in 1:max_iter
        for j in 1:k
            wf_copy!(X[j], V[j])
            X[j].f .*= 1 / max(vals[j], 1e-14)
            _cg(X[j], DdagD, V[j]; eps=cg_eps, maxsteps=cg_steps, verbose=vb)
        end

        mgs!(X)
        for j in 1:k
            mul!(AV[j], DdagD, X[j])
        end
        vals_new, psi, Apsi = rayleigh_ritz_k(X, AV, k)
        for j in 1:k
            wf_copy!(V[j],  psi[j])
            wf_copy!(AV[j], Apsi[j])
        end
        vals = vals_new

        max_res = 0.0
        for j in 1:k
            wf_copy!(r_tmp, AV[j])
            axpby!(-vals[j], V[j], 1, r_tmp)
            max_res = max(max_res, wf_norm(r_tmp))
        end
        verbose && @printf("[Inv.Iter. %3d]  λ₁ = %.12e   ‖r‖_max = %.3e\n",
            iter, vals[1], max_res)
        if max_res < tol
            println("Converged!")
            return vals, V
        end
    end
    @warn "max_iter=$max_iter 도달"
    return vals, V
end

println("block_inverse_iter! defined.")

block_inverse_iter! defined.


## 3. 게이지 배위 & CPU Dirac 연산자

In [8]:
U_cpu     = Initialize_Gaugefields(NC, Nwing, NX, NY, NZ, NT, condition="cold")
Ug        = load(GAUGE_FILE, "U")
substitute_U!(U_cpu, Ug)
x_tmpl    = Initialize_pseudofermion_fields(U_cpu[1], "Wilson", nowing=true)
D_cpu     = Dirac_operator(U_cpu, x_tmpl, params)
DdagD_cpu = DdagD_operator(U_cpu, x_tmpl, params)   # built-in CG 에 직접 전달

Ndof = NC * NX * NY * NZ * NT * 4
println("Gauge loaded.  DOF = $Ndof")
println("D†D 행렬 크기 = $(Ndof)² → 명시적 구성 없이 mul! 만 사용")

Gauge loaded.  DOF = 98304
D†D 행렬 크기 = 98304² → 명시적 구성 없이 mul! 만 사용


## 4. LMP 모델 → inexact 벡터 추출

In [9]:
layers = load(MODEL_FILE, "layers")
println("Model: $(length(layers)) layers,  last = $(typeof(layers[end]))")

function exchangeU!(M, U, para)
    for i in eachindex(M)
        M[i] isa PTLayer && (M[i].U = U)
    end
    M[end] isa Ritz_Layer && (M[end].D = Dirac_operator(U, x_tmpl, para))
end

exchangeU!(layers, U_cpu, params)
V_raw = layers[1:end-1](U_cpu)   # Vector{WF3x1} — inexact subspace
println("Inexact subspace: $(length(V_raw)) 벡터")

Model: 38 layers,  last = Ritz_Layer
Inexact subspace: 4 벡터


In [10]:
# GPU → CPU 변환 (필요시)
function to_cpu_wf(x, U_ref)
    x_cpu = Initialize_pseudofermion_fields(U_ref[1], "Wilson", nowing=true)
    substitute_fermion!(x_cpu, x)
    return x_cpu
end

V_init = try
    using CUDA
    [V_raw[i].f isa CuArray ? to_cpu_wf(V_raw[i], U_cpu) : deepcopy(V_raw[i])
     for i in eachindex(V_raw)]
catch
    [deepcopy(v) for v in V_raw]
end
println("V_init 준비 완료: $(length(V_init)) WF3x1")

V_init 준비 완료: 4 WF3x1


## 5. Ritz_Layer 캐시에서 inexact 기준선 확인

In [11]:
_ = layers[end](V_raw)
ritz_layer = layers[end]
ritz_vals_lmp = copy(ritz_layer._evals)
ritz_U_coeffs = copy(ritz_layer._evecs)
k = length(ritz_vals_lmp)

println("LMP Ritz values (D†D 의 inexact 근삿값):")
for i in 1:k
    @printf("  %2d: λ̃ = %.10e\n", i, ritz_vals_lmp[i])
end

LMP Ritz values (D†D 의 inexact 근삿값):
   1: λ̃ = 1.0926962853e-02
   2: λ̃ = 1.1077711075e-02
   3: λ̃ = 1.1960275294e-02
   4: λ̃ = 1.2491989537e-02


In [12]:
# Ritz 벡터 형성: ψ_m = Σ_j U_{jm} v_j
function form_ritz_vecs(V, C)
    nc = size(C, 2)
    psi = [deepcopy(V[1]) for _ in 1:nc]
    for m in 1:nc
        wf_zero!(psi[m])
        for j in 1:min(length(V), size(C,1))
            axpby!(C[j,m], V[j], 1, psi[m])
        end
        wf_normalize!(psi[m])
    end
    return psi
end

ritz_vecs_lmp = form_ritz_vecs(V_init, ritz_U_coeffs)

# Rayleigh quotient로 LMP Ritz 값 재확인
buf_chk = deepcopy(V_init[1])
Av_chk  = deepcopy(V_init[1])
println("Rayleigh quotient 검증 (LMP Ritz 값과 비교):")
for i in 1:k
    apply_DdagD!(Av_chk, D_cpu, ritz_vecs_lmp[i], buf_chk)
    rq = real(inner_product(ritz_vecs_lmp[i], Av_chk))
    @printf("  %2d:  RQ = %.8e   Ritz = %.8e   diff = %.2e\n",
        i, rq, ritz_vals_lmp[i], abs(rq - ritz_vals_lmp[i]))
end

Rayleigh quotient 검증 (LMP Ritz 값과 비교):
   1:  RQ = 1.09269629e-02   Ritz = 1.09269629e-02   diff = 2.62e-16
   2:  RQ = 1.10777111e-02   Ritz = 1.10777111e-02   diff = 3.12e-17
   3:  RQ = 1.19602753e-02   Ritz = 1.19602753e-02   diff = 2.88e-16
   4:  RQ = 1.24919895e-02   Ritz = 1.24919895e-02   diff = 2.78e-16


## 6. Preconditioned LOPCG — 내장 CG 전처리

탐색 방향을 잔차 W 대신 **(D†D)⁻¹·W** 로 교체.  
전처리 CG 는 느슨한 tolerance (`eps=1e-6`, 약 50~100 steps) 로도 충분.

| 방법 | 수렴률 (per iter) | 이유 |
|------|-----------------|------|
| 비전처리 LOPCG | `(gap)^iter` ≈ 0.994 | 스펙트럴 갭 작음 |
| **전처리 LOPCG** | `(κ_eff)^{-1/2}` | (D†D)⁻¹ 이 대역 모드 억제 |

`DdagD_cpu` (`DdagD_operator`) 를 그대로 전달 — 내부에서 `mul!` 사용.

In [13]:
V_lopcg = [deepcopy(v) for v in ritz_vecs_lmp]

println("=== Preconditioned LOPCG (내장 CG 전처리, warm-start from LMP) ===")
@time lopcg_vals, V_lopcg = precond_lopcg_wf!(DdagD_cpu, V_lopcg;
    max_iter   = 300,
    prec_eps   = 1e-6,
    prec_steps = 200,
    tol        = 1e-11,
    verbose    = true)

=== Preconditioned LOPCG (내장 CG 전처리, warm-start from LMP) ===
[P-LOPCG    1]  λ₁ = 7.8052124202e-03   ‖r‖_max = 5.227e-03
[P-LOPCG   10]  λ₁ = 6.8607036818e-03   ‖r‖_max = 2.018e-03
[P-LOPCG   20]  λ₁ = 6.7991179208e-03   ‖r‖_max = 9.749e-04
[P-LOPCG   30]  λ₁ = 6.7845039525e-03   ‖r‖_max = 1.262e-03
[P-LOPCG   40]  λ₁ = 6.7767544681e-03   ‖r‖_max = 1.346e-03
[P-LOPCG   50]  λ₁ = 6.7728085937e-03   ‖r‖_max = 7.790e-04
[P-LOPCG   60]  λ₁ = 6.7711534664e-03   ‖r‖_max = 6.708e-04
[P-LOPCG   70]  λ₁ = 6.7703190127e-03   ‖r‖_max = 8.206e-04
[P-LOPCG   80]  λ₁ = 6.7699322883e-03   ‖r‖_max = 4.715e-04
[P-LOPCG   90]  λ₁ = 6.7696899784e-03   ‖r‖_max = 4.297e-04
[P-LOPCG  100]  λ₁ = 6.7695425998e-03   ‖r‖_max = 4.102e-04
[P-LOPCG  110]  λ₁ = 6.7694436138e-03   ‖r‖_max = 3.780e-04
[P-LOPCG  120]  λ₁ = 6.7693709016e-03   ‖r‖_max = 3.576e-04
[P-LOPCG  130]  λ₁ = 6.7693179751e-03   ‖r‖_max = 3.003e-04
[P-LOPCG  140]  λ₁ = 6.7692796955e-03   ‖r‖_max = 2.844e-04
[P-LOPCG  150]  λ₁ = 6.7692497781e-03 

┌ Warning: max_iter=300 도달  (‖r‖ 확인 필요)
└ @ Main In[5]:90


([0.00676909850720286, 0.006816756002768939, 0.00698357349427215, 0.0071035953875491185], LatticeDiracOperators.Dirac_operators.WilsonFermion_4D_nowing{3}[LatticeDiracOperators.Dirac_operators.WilsonFermion_4D_nowing{3}(ComplexF64[-0.0025680634572382635 - 0.0011836539803161677im -0.0010339990954500524 - 0.000216297595526704im … -0.0002885963235959983 - 0.0011067307929824271im 0.00047899300491454944 - 0.0002910014848369476im; -0.0008700760215929068 + 0.0002745382643224683im -0.0020734389718994614 + 0.0009153017268832399im … -0.0010725824145781996 - 0.00019867731451940078im -0.0006182718509023642 - 0.0017938584774339441im; -0.0007042397256254332 - 0.003249389899996234im -0.0011117109131473907 - 0.0038316987299935597im … 0.002572549985491918 + 0.00018248441287643368im 0.002100226440040212 - 0.0016483801675685683im;;; -0.0007113595322705985 - 0.0008015456509659612im 0.0004387059323976088 - 0.00046862718513943774im … -0.0004980085924398115 - 0.00046665414737887716im -0.0008261793435435765 -

In [14]:
Av2  = deepcopy(V_lopcg[1])
buf2 = deepcopy(V_lopcg[1])
println("P-LOPCG residuals  ‖D†D ψ − λ ψ‖:")
for i in 1:k
    mul!(Av2, DdagD_cpu, V_lopcg[i])
    axpby!(-lopcg_vals[i], V_lopcg[i], 1, Av2)
    @printf("  %2d: λ = %.10e   ‖r‖ = %.3e\n", i, lopcg_vals[i], wf_norm(Av2))
end

P-LOPCG residuals  ‖D†D ψ − λ ψ‖:
   1: λ = 6.7690985072e-03   ‖r‖ = 5.265e-05
   2: λ = 6.8167560028e-03   ‖r‖ = 4.037e-05
   3: λ = 6.9835734943e-03   ‖r‖ = 7.895e-05
   4: λ = 7.1035953875e-03   ‖r‖ = 1.300e-04


## 7. Block Inverse Iteration — LatticeDiracOperators 내장 CG 사용

각 outer step:
1. `cg(X[j], DdagD, V[j]; eps=1e-24)` — LatticeDiracOperators 내장 CG 로 D†D Xⱼ = Vⱼ 를 풀기  
   warm-start: Xⱼ = Vⱼ/λⱼ (D†D·(ψ/λ) ≈ ψ → 초기 잔차 ≈ 0)
2. 블록 MGS 직교화 + Rayleigh-Ritz → 새 Ritz 쌍

| 파라미터 | 값 | 의미 |
|----------|----|------|
| `cg_eps` | `1e-24` | ‖r‖² < eps → ‖r‖ < 1e-12 |
| `cg_steps` | `5000` | CG 최대 반복 |
| `tol` | `1e-11` | outer 수렴 판정 기준 |

D†D 는 `DdagD_operator` 형태 그대로 — `cg` 내부에서 `mul!` 로 적용.

In [15]:
V_inv = [deepcopy(v) for v in V_lopcg]   # LOPCG warm-start

println("=== Block Inverse Iteration (LatticeDiracOperators 내장 cg) ===")
@time exact_vals, exact_vecs = block_inverse_iter!(DdagD_cpu, V_inv;
    max_iter  = 50,
    cg_eps    = 1e-24,
    cg_steps  = 5000,
    tol       = 1e-11,
    verbose   = true)

=== Block Inverse Iteration (LatticeDiracOperators 내장 cg) ===
[Inv.Iter.   1]  λ₁ = 6.769090213192e-03   ‖r‖_max = 1.532e-05
[Inv.Iter.   2]  λ₁ = 6.769084066914e-03   ‖r‖_max = 1.477e-05
[Inv.Iter.   3]  λ₁ = 6.769078999666e-03   ‖r‖_max = 1.425e-05
[Inv.Iter.   4]  λ₁ = 6.769074817845e-03   ‖r‖_max = 1.376e-05
[Inv.Iter.   5]  λ₁ = 6.769071363372e-03   ‖r‖_max = 1.328e-05
[Inv.Iter.   6]  λ₁ = 6.769068507029e-03   ‖r‖_max = 1.282e-05
[Inv.Iter.   7]  λ₁ = 6.769066143065e-03   ‖r‖_max = 1.237e-05
[Inv.Iter.   8]  λ₁ = 6.769064184840e-03   ‖r‖_max = 1.194e-05
[Inv.Iter.   9]  λ₁ = 6.769062561295e-03   ‖r‖_max = 1.152e-05
[Inv.Iter.  10]  λ₁ = 6.769061214086e-03   ‖r‖_max = 1.112e-05
[Inv.Iter.  11]  λ₁ = 6.769060095258e-03   ‖r‖_max = 1.074e-05
[Inv.Iter.  12]  λ₁ = 6.769059165354e-03   ‖r‖_max = 1.036e-05
[Inv.Iter.  13]  λ₁ = 6.769058391876e-03   ‖r‖_max = 1.000e-05
[Inv.Iter.  14]  λ₁ = 6.769057748031e-03   ‖r‖_max = 9.654e-06
[Inv.Iter.  15]  λ₁ = 6.769057211706e-03   ‖r‖_max = 9.3

┌ Warning: max_iter=50 도달
└ @ Main In[7]:56


([0.006769054484758887, 0.006816693164637774, 0.006983367073253713, 0.007102631854736023], LatticeDiracOperators.Dirac_operators.WilsonFermion_4D_nowing{3}[LatticeDiracOperators.Dirac_operators.WilsonFermion_4D_nowing{3}(ComplexF64[-0.002534915392030322 + 0.0012596788872463586im -0.0008152531658055598 + 0.0006713463160341989im … -0.0010490089898579108 - 0.0004788584438971638im 6.760718678374858e-5 - 0.0005649826653190623im; -0.00032858803947565185 + 0.0008440683752287973im -0.000587256728463008 + 0.0021997798850575422im … -0.0008339373697090249 + 0.0007072276636800722im -0.0017846591941530216 - 0.0006472290148858206im; -0.002992240213892037 - 0.0014993406922050071im -0.0036840507217274386 - 0.0015549733645467746im … 0.0017534830997152414 - 0.0019006173118989572im 2.215067040272093e-5 - 0.0026701043981516262im;;; -0.0010755656596877525 + 5.386008894106283e-5im -7.824413101275297e-5 - 0.0006238138933080525im … -0.0006785630200707664 + 8.998574590666697e-5im -0.0013955309109368388 - 4.585

## 8. 비교: LMP inexact vs LOPCG vs Block Inv.Iter.

In [16]:
n_show = min(k, length(exact_vals))

println("\n", "="^80)
@printf("%-4s  %-22s  %-22s  %-22s\n",
    "idx", "LMP Ritz (inexact)", "LOPCG", "Block Inv.Iter. (exact)")
println("-"^80)
for i in 1:n_show
    @printf("%3d   %20.12e  %20.12e  %20.12e\n",
        i, ritz_vals_lmp[i], lopcg_vals[i], exact_vals[i])
end
println("="^80)

println("\nLMP vs exact 상대 오차:")
for i in 1:n_show
    rel = abs(ritz_vals_lmp[i] - exact_vals[i]) / (abs(exact_vals[i]) + 1e-30)
    @printf("  %2d: %.3e\n", i, rel)
end


idx   LMP Ritz (inexact)      LOPCG                   Block Inv.Iter. (exact)
--------------------------------------------------------------------------------
  1     1.092696285335e-02    6.769098507203e-03    6.769054484759e-03
  2     1.107771107529e-02    6.816756002769e-03    6.816693164638e-03
  3     1.196027529409e-02    6.983573494272e-03    6.983367073254e-03
  4     1.249198953655e-02    7.103595387549e-03    7.102631854736e-03

LMP vs exact 상대 오차:
   1: 6.143e-01
   2: 6.251e-01
   3: 7.127e-01
   4: 7.588e-01


## 9. 최종 residual & 직교성 검증

In [17]:
buf3 = deepcopy(exact_vecs[1])
Av3  = deepcopy(exact_vecs[1])
n_show   = min(k, length(exact_vals))
exact_res = zeros(n_show)

println("최종 residual  ‖D†D ψ − λ ψ‖:")
for i in 1:n_show
    apply_DdagD!(Av3, D_cpu, exact_vecs[i], buf3)
    axpby!(-exact_vals[i], exact_vecs[i], 1, Av3)
    exact_res[i] = wf_norm(Av3)
    @printf("  k=%2d: λ = %14.10e   ‖r‖ = %.3e\n", i, exact_vals[i], exact_res[i])
end

println("\n직교성 ⟨ψ_i|ψ_j⟩ (대각=1, 비대각≈0):")
for i in 1:n_show
    for j in 1:n_show
        c = inner_product(exact_vecs[i], exact_vecs[j])
        @printf("%7.4f+%7.4fi  ", real(c), imag(c))
    end
    println()
end

최종 residual  ‖D†D ψ − λ ψ‖:
  k= 1: λ = 6.7690544848e-03   ‖r‖ = 6.226e-08
  k= 2: λ = 6.8166931646e-03   ‖r‖ = 1.184e-07
  k= 3: λ = 6.9833670733e-03   ‖r‖ = 5.913e-07
  k= 4: λ = 7.1026318547e-03   ‖r‖ = 2.695e-06

직교성 ⟨ψ_i|ψ_j⟩ (대각=1, 비대각≈0):
 1.0000+ 0.0000i   0.0000+-0.0000i  -0.0000+-0.0000i  -0.0000+-0.0000i  
 0.0000+ 0.0000i   1.0000+ 0.0000i   0.0000+-0.0000i  -0.0000+ 0.0000i  
-0.0000+ 0.0000i   0.0000+ 0.0000i   1.0000+ 0.0000i   0.0000+ 0.0000i  
-0.0000+ 0.0000i  -0.0000+-0.0000i   0.0000+-0.0000i   1.0000+ 0.0000i  


## 10. 저장

In [18]:
save(OUTPUT_FILE,
    "eigenvalues",       exact_vals[1:n_show],
    "eigenvectors",      exact_vecs[1:n_show],
    "residuals",         exact_res[1:n_show],
    "ritz_vals_lmp",     ritz_vals_lmp,
    "gauge_file",        GAUGE_FILE,
    "kappa",             params["κ"],
)
println("Saved $n_show exact eigenpairs → $OUTPUT_FILE")

Saved 4 exact eigenpairs → exact_lowmodes_config1.jld2


## 11. (선택) 여러 게이지 배위 일괄 처리

In [19]:
# GAUGE_DIR  = "C:/Users/_s2220186/Desktop/Julia/gauge"
# config_ids = 1:10
#
# for cid in config_ids
#     gfile = joinpath(GAUGE_DIR, "config_$cid.jld2")
#     !isfile(gfile) && continue
#
#     U_i = Initialize_Gaugefields(NC, Nwing, NX, NY, NZ, NT, condition="cold")
#     substitute_U!(U_i, load(gfile, "U"))
#     D_i = Dirac_operator(U_i, x_tmpl, params)
#
#     exchangeU!(layers, U_i, params)
#     V_i   = layers[1:end-1](U_i)
#     _     = layers[end](V_i)
#     psi_i = form_ritz_vecs(
#         [to_cpu_wf(v, U_i) for v in V_i],
#         copy(layers[end]._evecs))
#
#     lopcg_v_i, psi_i = lopcg_wf!(D_i, psi_i; tol=1e-11, verbose=false)
#     ev_i, vecs_i, res_i = lanczos_rr(D_i, psi_i[1], krylov_dim, k; verbose=false)
#
#     save("exact_config_$cid.jld2",
#         "eigenvalues", ev_i, "eigenvectors", vecs_i, "residuals", res_i)
#     @printf("config %2d: λ_min = %.6e\n", cid, ev_i[1])
# end

println("(multi-config 셀: 주석 해제 후 실행)")

(multi-config 셀: 주석 해제 후 실행)
